In [ ]:
# ============================================
# 1. Установка (Colab): timm, foolbox, torchao — эксперименты QAT (не PTQ)
# ============================================
!pip install timm foolbox torchao

# ============================================
# 2. Импорты
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

import timm
from timm.data import create_transform, resolve_data_config

import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import platform
import hashlib
import uuid
from datetime import datetime, timezone
import random
import json
import logging
from tqdm import tqdm
import foolbox as fb
import subprocess
import zipfile
import shutil

# QAT (torchao): int8 dynamic activation + int4 weight, prepare → train → convert
from torchao.quantization.qat import Int8DynActInt4WeightQATQuantizer

# ============================================
# 2.5. Настройка окружения Hugging Face
# ============================================
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_TOKEN"] = ""  # для публичных моделей токен не нужен

# ============================================
# 3. Конфигурация: общая часть + 3 независимых варианта прогона
# ============================================
CONFIG_SHARED = {
    'seed': 42,
    'data_root': 'tiny-imagenet-200',
    'batch_size': 64,
    'num_workers': 4,
    'pin_memory': True,
    'prefetch_factor': 2,
    'persistent_workers': True,
    'num_epochs': 10,
    'lr_head': 1e-4,
    'lr_base': 2e-5,
    'weight_decay': 0.05,
    'weight_decay_head': 0.01,
    'warmup_epochs': 3,
    'early_stopping_patience': 3,
    'use_amp': True,
    'tf32': True,
    'epsilons': [0.001, 0.005, 0.01, 0.03, 0.05],
    'use_subset': False,
    'train_subset_size': 500,
    'val_subset_size': 200,
    'model_name': 'vit_small_patch16_224.augreg_in21k_ft_in1k',
    'num_classes': 200,
    'hybrid_quantize_blocks': (6, 12),
    'qat_groupsize': 32,
    'qat_num_epochs': 10,
    'qat_padding_allowed': False,
    'save_models': True,
    'models_dir': './saved_models_tinyimagenet_adv_qat',
    'repro_artifacts': True,
    'repro_run_id': None,
    'repro_git_cwd': None,
    'adversarial_training': True,
    'adv_epsilon': 0.01,
    'adv_alpha': 0.005,
    'adv_steps': 5,
    'adv_random_start': True,
    # Второй полный цикл FP32+QAT без adversarial training (бейзлайн); False — только adv-цикл
    'run_baseline_no_adv': True,
    # Синхронизация с GitHub
    'github_sync_enabled': True,
    'github_repo_https': 'https://github.com/Vasil8/Quantized-ViT-Robustness.git',
    'github_branch': 'main',
    'github_token_secret_name': 'GITHUB_TOKEN',
    'github_repo_local_dir': '/content/Quantized-ViT-Robustness',
}

# Бейзлайн: обучение без adversarial training (сравнение влияния квантования на робастность)
CONFIG_BASELINE_NO_ADV = dict(CONFIG_SHARED)
CONFIG_BASELINE_NO_ADV['adversarial_training'] = False
CONFIG_BASELINE_NO_ADV['models_dir'] = './saved_models_tinyimagenet_baseline_no_adv'

CONFIG_VARIANT_FP32 = {
    'experiment_variant': 'fp32',
    'run_finetune': True,
    'fp32_checkpoint_path': None,
    'evaluate_branches': ('fp32',),
}

CONFIG_VARIANT_FULL_QAT = {
    'experiment_variant': 'full_qat',
    'run_finetune': False,
    'run_qat_finetune': True,
    'finetune_replace_head': False,
    'fp32_checkpoint_path': None,
    'evaluate_branches': ('full_qat',),
    'best_checkpoint_filename': 'best_model_qat_full.pth',
}

CONFIG_VARIANT_HYBRID_QAT = {
    'experiment_variant': 'hybrid_qat',
    'run_finetune': False,
    'run_qat_finetune': True,
    'finetune_replace_head': False,
    'fp32_checkpoint_path': None,
    'evaluate_branches': ('hybrid_qat',),
    'best_checkpoint_filename': 'best_model_qat_hybrid.pth',
}

def merge_experiment_config(shared: dict, variant: dict) -> dict:
    out = dict(shared)
    out.update(variant)
    return out

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================
# 4. Фиксация seed
# ============================================
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CONFIG_SHARED['seed'])

# ============================================
# 4.5. Артефакты воспроизводимости (отдельный прогон → отдельная папка + манифесты)
# ============================================
def _json_safe(obj):
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    return str(obj)

def config_sha256(cfg_dict):
    payload = json.dumps(_json_safe(cfg_dict), sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def collect_environment_info():
    import importlib.metadata as ilm
    pkgs = ['torch', 'torchvision', 'timm', 'foolbox', 'torchao', 'numpy', 'eagerpy', 'pandas']
    versions = {}
    for p in pkgs:
        try:
            versions[p] = ilm.version(p)
        except Exception:
            versions[p] = None
    info = {
        'python': sys.version.split('\n')[0],
        'platform': platform.platform(),
        'packages': versions,
    }
    if torch.cuda.is_available():
        info['cuda'] = {
            'torch_cuda_version': torch.version.cuda,
            'device_name': torch.cuda.get_device_name(0),
            'cudnn_enabled': torch.backends.cudnn.enabled,
            'cudnn_version': torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
        }
    else:
        info['cuda'] = None
    return info

def collect_git_provenance(cwd=None):
    try:
        r = subprocess.run(
            ['git', 'rev-parse', 'HEAD'],
            capture_output=True, text=True, timeout=8, cwd=cwd
        )
        if r.returncode != 0:
            return {'available': False, 'reason': (r.stderr or 'not a git repository').strip()}
        commit = r.stdout.strip()
        st = subprocess.run(
            ['git', 'status', '--porcelain'],
            capture_output=True, text=True, timeout=8, cwd=cwd
        )
        dirty = len(st.stdout.strip()) > 0
        br = subprocess.run(
            ['git', 'rev-parse', '--abbrev-ref', 'HEAD'],
            capture_output=True, text=True, timeout=8, cwd=cwd
        )
        branch = br.stdout.strip() if br.returncode == 0 else None
        return {'available': True, 'commit': commit, 'branch': branch, 'working_tree_dirty': dirty}
    except FileNotFoundError:
        return {'available': False, 'reason': 'git executable not found'}
    except Exception as e:
        return {'available': False, 'reason': str(e)}

def prepare_experiment_run_directory(config):
    base = config['models_dir']
    os.makedirs(base, exist_ok=True)
    if not config.get('repro_artifacts', True):
        config['models_base_dir'] = base
        config['experiment_run_id'] = None
        config['config_fingerprint_sha256'] = config_sha256(_json_safe({k: v for k, v in config.items()}))
        return base
    rid = config.get('repro_run_id')
    if not rid:
        ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        cfg_minus = {k: v for k, v in config.items() if k not in ('repro_run_id', 'experiment_run_id', 'models_base_dir', 'config_fingerprint_sha256')}
        fp = config_sha256(_json_safe(cfg_minus))[:12]
        rid = f"{ts}__{fp}__{uuid.uuid4().hex[:6]}"
    safe = rid.replace(os.sep, '_').replace(':', '-')
    run_dir = os.path.join(base, safe)
    os.makedirs(run_dir, exist_ok=True)
    config['models_base_dir'] = base
    config['models_dir'] = run_dir
    config['experiment_run_id'] = safe
    config['config_fingerprint_sha256'] = config_sha256(
        _json_safe({k: v for k, v in config.items() if k not in ('experiment_run_id', 'models_base_dir')})
    )
    logger.info(f"Каталог артефактов прогона: {run_dir}")
    return run_dir

def reset_models_dir_base(config, models_dir_base):
    config['models_dir'] = models_dir_base
    for k in ('experiment_run_id', 'models_base_dir', 'config_fingerprint_sha256'):
        config.pop(k, None)
    config['repro_run_id'] = None
    return config

def _in_colab_runtime():
    try:
        import google.colab
        return True
    except ImportError:
        return False

def _github_repo_url_with_token(https_url: str, token: str) -> str:
    if not https_url.startswith('https://'):
        return https_url
    rest = https_url[len('https://'):]
    return f'https://x-access-token:{token}@{rest}'

def _copy_tree_merge(src: str, dst: str) -> None:
    for root, _dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        ddir = dst if rel == '.' else os.path.join(dst, rel)
        os.makedirs(ddir, exist_ok=True)
        for fn in files:
            shutil.copy2(os.path.join(root, fn), os.path.join(ddir, fn))

def get_github_token():
    t = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
    if t:
        return t.strip()
    if _in_colab_runtime():
        try:
            from google.colab import userdata
            name = CONFIG_SHARED.get('github_token_secret_name', 'GITHUB_TOKEN')
            return userdata.get(name).strip()
        except Exception:
            return None
    return None

def sync_artifacts_to_github(models_dir_base: str, experiment_run_dirs: list) -> None:
    if not CONFIG_SHARED.get('github_sync_enabled'):
        logger.info('GitHub sync выключен.')
        return
    if CONFIG_SHARED.get('github_sync_only_in_colab') and not _in_colab_runtime():
        logger.info('GitHub sync пропущен вне Colab.')
        return
    token = get_github_token()
    if not token:
        logger.warning('GitHub sync пропущен: нет токена.')
        return
    repo_https = CONFIG_SHARED['github_repo_https']
    branch = CONFIG_SHARED.get('github_branch', 'main')
    local_dir = os.path.expanduser(CONFIG_SHARED.get('github_repo_local_dir', '/content/Quantized-ViT-Robustness'))
    if not shutil.which('git'):
        logger.error('git не найден.')
        return
    authed = _github_repo_url_with_token(repo_https, token)
    git_dir = os.path.join(local_dir, '.git')
    try:
        if not os.path.isdir(git_dir):
            parent = os.path.dirname(os.path.abspath(local_dir))
            if parent:
                os.makedirs(parent, exist_ok=True)
            subprocess.run(
                ['git', 'clone', '--branch', branch, '--single-branch', authed, local_dir],
                check=True, timeout=600,
            )
        else:
            subprocess.run(['git', '-C', local_dir, 'remote', 'set-url', 'origin', authed], check=True, timeout=30)
            subprocess.run(['git', '-C', local_dir, 'pull', '--ff-only', 'origin', branch], check=True, timeout=300)
    except subprocess.CalledProcessError as e:
        logger.error('git clone/pull не удался: %s', e)
        return
    colab_art = os.path.join(local_dir, 'colab_artifacts')
    os.makedirs(colab_art, exist_ok=True)
    dest_root = os.path.join(colab_art, os.path.basename(os.path.abspath(models_dir_base)))
    abs_base = os.path.abspath(models_dir_base)
    if os.path.isdir(abs_base):
        _copy_tree_merge(abs_base, dest_root)
    manifest = {
        'updated_utc': datetime.now(timezone.utc).isoformat(),
        'models_dir_base': abs_base,
        'experiment_run_dirs': [os.path.abspath(d) for d in experiment_run_dirs],
    }
    with open(os.path.join(colab_art, 'last_sync_manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    try:
        subprocess.run(['git', '-C', local_dir, 'config', 'user.email', 'colab-sync@users.noreply.github.com'], check=False)
        subprocess.run(['git', '-C', local_dir, 'config', 'user.name', 'Colab Sync'], check=False)
        subprocess.run(['git', '-C', local_dir, 'add', '-A'], check=True, timeout=300)
        st = subprocess.run(['git', '-C', local_dir, 'status', '--porcelain'], capture_output=True, text=True, timeout=30)
        if not (st.stdout or '').strip():
            logger.info('Нет изменений для коммита.')
            return
        msg = 'colab artifacts sync ' + manifest['updated_utc']
        subprocess.run(['git', '-C', local_dir, 'commit', '-m', msg], check=True, timeout=120)
        subprocess.run(['git', '-C', local_dir, 'remote', 'set-url', 'origin', authed], check=True, timeout=30)
        subprocess.run(['git', '-C', local_dir, 'push', 'origin', branch], check=True, timeout=600)
        logger.info('Артефакты отправлены в GitHub.')
    except subprocess.CalledProcessError as e:
        logger.error('git commit/push не удался: %s', e)

def save_reproducibility_artifacts(
    config,
    dataset_sizes,
    device,
    results,
    df_results=None,
    df_metrics_summary=None,
    df_asr=None,
    df_gap=None,
):
    out_dir = config['models_dir']
    os.makedirs(out_dir, exist_ok=True)
    cfg_snapshot = _json_safe({k: v for k, v in config.items()})
    env = collect_environment_info()
    git_cwd = config.get('repro_git_cwd') or os.getcwd()
    git_p = collect_git_provenance(cwd=git_cwd)

    results_out = {}
    for model_name, d in results.items():
        results_out[model_name] = {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items()}

    bundle = {
        'schema': 'tinyimagenet_adv_qat_v1',
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'experiment_run_id': config.get('experiment_run_id'),
        'models_base_dir': config.get('models_base_dir'),
        'run_directory': out_dir,
        'config_fingerprint_sha256': config.get('config_fingerprint_sha256'),
        'config': cfg_snapshot,
        'dataset_sizes': dict(dataset_sizes),
        'device': str(device),
        'environment': env,
        'git': git_p,
        'results': results_out,
        'evaluation_note': 'QAT torchao: для квантованных моделей используется суррогат (prepare) без FP32.',
    }
    if df_metrics_summary is not None:
        bundle["metrics_summary"] = df_metrics_summary.round(6).to_dict(orient="index")
    if df_asr is not None:
        bundle["attack_success_rate_by_eps"] = df_asr.round(6).to_dict(orient="index")
    if df_gap is not None:
        bundle["gap_clean_minus_robust_by_eps"] = df_gap.round(6).to_dict(orient="index")
    path_bundle = os.path.join(out_dir, 'experiment_bundle.json')
    with open(path_bundle, 'w', encoding='utf-8') as f:
        json.dump(bundle, f, indent=2, ensure_ascii=False)
    with open(os.path.join(out_dir, 'environment.json'), 'w', encoding='utf-8') as f:
        json.dump(env, f, indent=2, ensure_ascii=False)
    with open(os.path.join(out_dir, 'config_resolved.json'), 'w', encoding='utf-8') as f:
        json.dump(cfg_snapshot, f, indent=2, ensure_ascii=False)
    with open(os.path.join(out_dir, 'git_provenance.json'), 'w', encoding='utf-8') as f:
        json.dump(git_p, f, indent=2, ensure_ascii=False)
    with open(os.path.join(out_dir, 'results.json'), 'w', encoding='utf-8') as f:
        json.dump(results_out, f, indent=2, ensure_ascii=False)
    if df_results is not None:
        df_results.round(6).to_csv(os.path.join(out_dir, 'results_table.csv'), encoding='utf-8')
    if df_metrics_summary is not None:
        df_metrics_summary.round(6).to_csv(os.path.join(out_dir, 'results_metrics_summary.csv'), encoding='utf-8')
    if df_asr is not None:
        df_asr.round(6).to_csv(os.path.join(out_dir, 'results_attack_success_rate.csv'), encoding='utf-8')
    if df_gap is not None:
        df_gap.round(6).to_csv(os.path.join(out_dir, 'results_clean_robust_gap.csv'), encoding='utf-8')
    lines = [
        f"run_id: {config.get('experiment_run_id')}",
        f"config_sha256: {config.get('config_fingerprint_sha256')}",
        f"bundle: {path_bundle}",
        f"device: {device}",
    ]
    if git_p.get('available'):
        lines.append(f"git_commit: {git_p.get('commit')} dirty={git_p.get('working_tree_dirty')}")
    with open(os.path.join(out_dir, 'RUN_SUMMARY.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    logger.info(f"Сохранены артефакты воспроизводимости в {out_dir}")

# ============================================
# 5. Загрузка данных Tiny ImageNet
# ============================================
def download_and_prepare_tiny_imagenet(root_dir):
    train_dir = os.path.join(root_dir, 'train')
    val_dir = os.path.join(root_dir, 'val')

    if os.path.exists(train_dir) and os.path.exists(val_dir):
        logger.info("Tiny ImageNet уже загружен и подготовлен.")
        return train_dir, val_dir

    os.makedirs(root_dir, exist_ok=True)
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    zip_path = os.path.join(root_dir, "tiny-imagenet-200.zip")

    if not os.path.exists(zip_path):
        logger.info("Скачивание Tiny ImageNet...")
        subprocess.run(["wget", "-O", zip_path, url], check=True)
    else:
        logger.info("Zip-архив уже существует, пропускаем скачивание.")

    logger.info("Распаковка...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(root_dir)

    extracted_dir = os.path.join(root_dir, 'tiny-imagenet-200')
    if os.path.exists(extracted_dir):
        for item in os.listdir(extracted_dir):
            os.rename(os.path.join(extracted_dir, item), os.path.join(root_dir, item))
        os.rmdir(extracted_dir)

    val_dir_raw = os.path.join(root_dir, 'val')
    val_images_dir = os.path.join(val_dir_raw, 'images')
    val_annotations = os.path.join(val_dir_raw, 'val_annotations.txt')

    if os.path.exists(val_images_dir) and os.path.exists(val_annotations):
        logger.info("Подготовка валидационной выборки...")
        with open(val_annotations, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    img_name, class_id = parts[0], parts[1]
                    src = os.path.join(val_images_dir, img_name)
                    dst_dir = os.path.join(val_dir_raw, class_id)
                    os.makedirs(dst_dir, exist_ok=True)
                    dst = os.path.join(dst_dir, img_name)
                    if os.path.exists(src):
                        os.rename(src, dst)
        if os.path.exists(val_images_dir):
            os.rmdir(val_images_dir)

    logger.info("Подготовка завершена.")
    return train_dir, os.path.join(root_dir, 'val')

def download_and_prepare_data(config):
    train_dir, val_dir = download_and_prepare_tiny_imagenet(config['data_root'])

    temp_model = timm.create_model(config['model_name'], pretrained=True)
    data_config = resolve_data_config(temp_model.pretrained_cfg, model=temp_model)
    del temp_model

    train_transform = create_transform(**data_config, is_training=True)
    val_transform = create_transform(**data_config, is_training=False)

    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    val_dataset = ImageFolder(root=val_dir, transform=val_transform)

    if config['use_subset']:
        train_dataset = Subset(train_dataset, list(range(min(config['train_subset_size'], len(train_dataset)))))
        val_dataset = Subset(val_dataset, list(range(min(config['val_subset_size'], len(val_dataset)))))

    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=config['num_workers'],
        pin_memory=config['pin_memory'] and torch.cuda.is_available(),
        persistent_workers=config['persistent_workers'] if config['num_workers'] > 0 else False,
        prefetch_factor=config['prefetch_factor'] if config['num_workers'] > 0 else None,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        pin_memory=config['pin_memory'] and torch.cuda.is_available(),
        persistent_workers=False,
        prefetch_factor=config['prefetch_factor'] if config['num_workers'] > 0 else None,
    )

    dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
    logger.info(f"Размер обучающей выборки: {dataset_sizes['train']}, валидационной: {dataset_sizes['val']}")

    mean = torch.tensor(data_config['mean']).view(1, 3, 1, 1)
    std = torch.tensor(data_config['std']).view(1, 3, 1, 1)

    return train_loader, val_loader, dataset_sizes, mean, std

# ============================================
# 6. Функция PGD-атаки (нормализованное пространство, границы = прообраз [0,1])
# ============================================
def _normalized_input_bounds(mean, std, device):
    mean_d = mean.to(device)
    std_d = std.to(device)
    lo = (0.0 - mean_d) / std_d
    hi = (1.0 - mean_d) / std_d
    return lo, hi

def pgd_attack(model, images, labels, epsilon, alpha, steps, criterion, mean, std, device,
               random_start=True):
    was_training = model.training
    model.eval()
    lo, hi = _normalized_input_bounds(mean, std, device)

    adv_images = images.clone().detach()
    if random_start:
        adv_images = adv_images + torch.empty_like(adv_images).uniform_(-epsilon, epsilon)
        eta0 = torch.clamp(adv_images - images, min=-epsilon, max=epsilon)
        adv_images = torch.clamp(images + eta0, min=lo, max=hi)
    adv_images = adv_images.requires_grad_(True)

    for _ in range(steps):
        outputs = model(adv_images)
        loss = criterion(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False)[0]
        adv_images = adv_images + alpha * grad.sign()
        eta = torch.clamp(adv_images - images, min=-epsilon, max=epsilon)
        adv_images = torch.clamp(images + eta, min=lo, max=hi)
        adv_images = adv_images.detach().requires_grad_(True)

    if was_training:
        model.train()
    return adv_images.detach()

# ============================================
# 7. Функция fine-tuning с adversarial training
# ============================================
def finetune_model(model, num_classes, dataloaders, dataset_sizes, config, device, mean, std):
    if config.get('finetune_replace_head', True):
        if hasattr(model, 'head'):
            in_features = model.head.in_features
            model.head = nn.Linear(in_features, num_classes)
        elif hasattr(model, 'reset_classifier'):
            model.reset_classifier(num_classes)
        else:
            raise AttributeError("Модель не имеет ни 'head', ни 'reset_classifier'")
    else:
        logger.info("Голова не пересоздаётся (режим QAT после загрузки чекпойнта).")

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    head_params = list(model.head.parameters())
    base_params = [p for n, p in model.named_parameters() if not n.startswith('head.')]

    optimizer = optim.AdamW([
        {'params': base_params, 'lr': config['lr_base'], 'weight_decay': config['weight_decay']},
        {'params': head_params, 'lr': config['lr_head'], 'weight_decay': config['weight_decay_head']}
    ])

    def warmup_cosine(epoch, warmup=config['warmup_epochs'], total=config['num_epochs']):
        if epoch < warmup:
            return float(epoch + 1) / warmup
        else:
            progress = (epoch - warmup) / (total - warmup)
            return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    use_amp_cuda = bool(config['use_amp'] and device.type == 'cuda')
    _amp_dev = device.type if device.type in ('cuda', 'cpu') else 'cpu'
    scaler = torch.amp.GradScaler(_amp_dev, enabled=use_amp_cuda)

    best_model_wts = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_acc = 0.0
    epochs_no_improve = 0

    if config['save_models']:
        os.makedirs(config['models_dir'], exist_ok=True)
        ckpt_name = config.get('best_checkpoint_filename', 'best_model_fp32.pth')
        best_model_path = os.path.join(config['models_dir'], ckpt_name)

    for epoch in range(config['num_epochs']):
        logger.info(f'Epoch {epoch+1}/{config["num_epochs"]}')
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in tqdm(dataloaders[phase], desc=phase):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    if phase == 'train' and config['adversarial_training']:
                        inputs_adv = pgd_attack(
                            model, inputs, labels,
                            config['adv_epsilon'],
                            config['adv_alpha'],
                            config['adv_steps'],
                            criterion,
                            mean, std, device,
                            random_start=config.get('adv_random_start', True),
                        )
                        with torch.amp.autocast(device.type, enabled=use_amp_cuda):
                            outputs = model(inputs_adv)
                            loss = criterion(outputs, labels)
                    else:
                        with torch.amp.autocast(device.type, enabled=use_amp_cuda):
                            outputs = model(inputs)
                            loss = criterion(outputs, labels)

                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            logger.info(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    epochs_no_improve = 0
                    if config['save_models']:
                        torch.save(best_model_wts, best_model_path)
                        logger.info(f"Сохранена лучшая модель с accuracy = {best_acc:.4f}")
                else:
                    epochs_no_improve += 1

        scheduler.step()

        if epochs_no_improve >= config['early_stopping_patience']:
            logger.info(f"Ранняя остановка на эпохе {epoch+1}")
            break

    logger.info(f'Лучшая валидационная точность: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

# ============================================
# 8. Вспомогательные функции для квантования и оценки
# ============================================
def state_dict_fully_detached(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def load_fp32_checkpoint_as_state_dict(path):
    try:
        blob = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        blob = torch.load(path, map_location='cpu')
    if isinstance(blob, dict) and 'state_dict' in blob:
        blob = blob['state_dict']
    return {k: v.detach().cpu().clone() if torch.is_tensor(v) else v for k, v in blob.items()}

def resolve_eval_branch_labels(config):
    lo, hi = config['hybrid_quantize_blocks']
    return {
        'fp32': 'FP32',
        'full_qat': 'Full QAT (int8 act int4 w)',
        'hybrid_qat': f'Hybrid QAT (blocks {lo}-{hi - 1})',
    }

def build_fp32_model_from_weights(config, state_dict):
    m = timm.create_model(
        config['model_name'],
        pretrained=False,
        num_classes=config['num_classes'],
    )
    m.load_state_dict(state_dict, strict=True)
    return m

def get_qat_quantizer(config):
    return Int8DynActInt4WeightQATQuantizer(
        groupsize=config.get('qat_groupsize', 32),
        padding_allowed=config.get('qat_padding_allowed', False),
    )

def convert_hybrid_qat_blocks(m_prepared, qat_q, lo, hi):
    """Конвертирует только подготовленные QAT-блоки (prepare был на blocks[lo:hi]), не всю модель."""
    for i in range(lo, hi):
        qat_q.convert(m_prepared.blocks[i])

def save_state_dict_size_mb(model, path):
    """Сохраняет state_dict и возвращает размер файла в МБ (реальный размер на диске)."""
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / (1024 * 1024)

def print_model_size(model, name, models_dir='./'):
    device = next(model.parameters()).device
    model.cpu()
    path = os.path.join(models_dir, f"{name}.pth")
    torch.save(model.state_dict(), path)
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"Size of {name}: {size:.2f} MB")
    model.to(device)

def get_fmodel(model, mean, std, device):
    preprocessing = dict(mean=mean.cpu().numpy().flatten(), std=std.cpu().numpy().flatten(), axis=-3)
    fmodel = fb.PyTorchModel(model.eval(), bounds=(0, 1), device=device, preprocessing=preprocessing)
    return fmodel

def model_supports_fgsm_input_grad(model, device, spatial_size=224):
    model.eval()
    x = torch.zeros(1, 3, spatial_size, spatial_size, device=device, requires_grad=True)
    try:
        out = model(x)
        loss = out.float().sum()
        loss.backward()
        g = x.grad
        return g is not None and torch.isfinite(g).all() and g.abs().sum() > 0
    finally:
        model.zero_grad(set_to_none=True)

def evaluate_clean_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Clean evaluation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

def evaluate_adversarial_accuracy(target_model, attack_model, dataloader, epsilon, mean, std, device):
    target_model.eval()
    attack_model.eval()

    fmodel_attack = get_fmodel(attack_model, mean, std, device)
    fmodel_target = get_fmodel(target_model, mean, std, device)

    attack = fb.attacks.FGSM()

    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc=f"Adversarial (eps={epsilon})"):
        images, labels = images.to(device), labels.to(device)

        mean_img = mean.to(device)
        std_img = std.to(device)
        images_denorm = images * std_img + mean_img

        _, clipped, _ = attack(fmodel_attack, images_denorm, labels, epsilons=[epsilon])
        if isinstance(clipped, (list, tuple)):
            clipped = clipped[0]
        elif hasattr(clipped, "ndim") and clipped.ndim == 5:
            clipped = clipped[0]

        logits = fmodel_target(clipped)
        _, predicted = torch.max(logits, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return correct / total

def compute_robustness_metric_tables(results, epsilons):
    import pandas as pd
    summary_rows = []
    asr_records = []
    gap_records = []
    idx = []
    for model_name, d in results.items():
        clean = float(d["clean"])
        robust = np.array([float(d[f"eps={e}"]) for e in epsilons], dtype=np.float64)
        asr = 1.0 - robust
        gap = clean - robust
        worst_i = int(np.argmin(robust))
        summary_rows.append(
            {
                "clean_acc": clean,
                "mean_robust_acc": float(robust.mean()),
                "std_robust_acc_across_eps": float(robust.std(ddof=0)),
                "mean_ASR": float(asr.mean()),
                "min_robust_acc": float(robust.min()),
                "worst_case_eps": float(epsilons[worst_i]),
                "mean_gap_clean_minus_robust": float(gap.mean()),
            }
        )
        row_asr = {"clean_acc": clean}
        row_gap = {"clean_acc": clean}
        for i, eps in enumerate(epsilons):
            row_asr[f"ASR_eps={eps}"] = float(asr[i])
            row_gap[f"gap_eps={eps}"] = float(gap[i])
        idx.append(model_name)
        asr_records.append(row_asr)
        gap_records.append(row_gap)
    df_summary = pd.DataFrame(summary_rows, index=idx)
    df_asr = pd.DataFrame(asr_records, index=idx)
    df_gap = pd.DataFrame(gap_records, index=idx)
    return df_summary, df_asr, df_gap

# ============================================
# 9. Основная функция эксперимента (исправленная, без FP32-суррогата)
# ============================================
def prepare_device_for_training(config):
    if torch.cuda.is_available():
        if config.get('tf32'):
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        device = torch.device("cuda")
        logger.info(f"Используется GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        logger.info("Используется CPU")
    return device

def run_experiment_pipeline(config, dataloaders, dataset_sizes, mean, std, device):
    """Один независимый прогон: свой run_dir, свои модели и метрики.
       Для QAT-веток создаётся квантованный суррогат из подготовленной (prepare) модели,
       которая сохраняет градиенты через STE. Конвертированная (convert) модель
       недифференцируема и не может служить суррогатом.
    """
    if device.type == 'cpu':
        config['use_amp'] = False

    prepare_experiment_run_directory(config)

    val_loader = dataloaders['val']

    branches = tuple(config['evaluate_branches'])
    labels = resolve_eval_branch_labels(config)
    unknown = set(branches) - {'fp32', 'full_qat', 'hybrid_qat'}
    if unknown:
        raise ValueError(f"evaluate_branches: неизвестные ключи {unknown}")

    # ---------- 1. Получение FP32 весов (через fine-tune или загрузку) ----------
    if config['run_finetune']:
        logger.info("Создание и fine-tuning базовой модели FP32...")
        model_fp32_trained = timm.create_model(config['model_name'], pretrained=True)
        model_fp32_trained = finetune_model(
            model_fp32_trained,
            num_classes=config['num_classes'],
            dataloaders=dataloaders,
            dataset_sizes=dataset_sizes,
            config=config,
            device=device,
            mean=mean,
            std=std,
        )
        logger.info("Fine-tuning завершён.")
        fp32_weights_snapshot = state_dict_fully_detached(model_fp32_trained)
        del model_fp32_trained
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        ckpt = config.get('fp32_checkpoint_path')
        if not ckpt or not os.path.isfile(ckpt):
            raise FileNotFoundError(
                f"Без fine-tune нужен существующий fp32_checkpoint_path (best_model_fp32.pth); сейчас: {ckpt!r}"
            )
        logger.info(f"Загрузка FP32 весов из {ckpt}")
        fp32_weights_snapshot = load_fp32_checkpoint_as_state_dict(ckpt)

    # ---------- 2. Базовая FP32 модель (для ветки fp32) ----------
    model_fp32 = build_fp32_model_from_weights(config, fp32_weights_snapshot).to(device)

    # ---------- 3. Словари для хранения моделей и суррогатов ----------
    models = {}          # целевые модели (для вычисления accuracy)
    surrogate_models = {}  # суррогаты для генерации атак (только для квантованных)

    # ---------- 4. Ветка FP32 ----------
    if 'fp32' in branches:
        models[labels['fp32']] = model_fp32
        # FP32 сама поддерживает градиенты, суррогат не нужен

    # ---------- 5. Ветка Full QAT ----------
    if 'full_qat' in branches:
        logger.info("QAT: prepare полной модели (Int8DynActInt4Weight)...")
        m_prepared = build_fp32_model_from_weights(config, fp32_weights_snapshot)
        qat_q = get_qat_quantizer(config)
        qat_q.prepare(m_prepared)   # подготавливаем (сохраняем градиенты)
        m_prepared = m_prepared.to(device)

        if config.get('run_qat_finetune', False):
            qcfg = dict(config)
            qcfg['num_epochs'] = config.get('qat_num_epochs', config['num_epochs'])
            m_prepared = finetune_model(
                m_prepared, config['num_classes'], dataloaders, dataset_sizes,
                qcfg, device, mean, std,
            )
        # Суррогат: та же архитектура + prepare (FakeQuant), затем веса из обученной prepared-модели
        surrogate_full = build_fp32_model_from_weights(config, fp32_weights_snapshot)
        qat_q.prepare(surrogate_full)
        surrogate_full.load_state_dict(m_prepared.state_dict())
        surrogate_full = surrogate_full.to(device)
        # Конвертируем целевую модель
        logger.info("QAT: convert (full)...")
        qat_q.convert(m_prepared)   # после convert градиентов нет
        models[labels['full_qat']] = m_prepared
        surrogate_models[labels['full_qat']] = surrogate_full

    # ---------- 6. Ветка Hybrid QAT ----------
    if 'hybrid_qat' in branches:
        logger.info("QAT: prepare гибрид (блоки hybrid_quantize_blocks)...")
        m_prepared = build_fp32_model_from_weights(config, fp32_weights_snapshot)
        qat_q = get_qat_quantizer(config)
        lo, hi = config['hybrid_quantize_blocks']
        for i in range(lo, hi):
            qat_q.prepare(m_prepared.blocks[i])
        m_prepared = m_prepared.to(device)

        if config.get('run_qat_finetune', False):
            qcfg = dict(config)
            qcfg['num_epochs'] = config.get('qat_num_epochs', config['num_epochs'])
            m_prepared = finetune_model(
                m_prepared, config['num_classes'], dataloaders, dataset_sizes,
                qcfg, device, mean, std,
            )
        surrogate_hybrid = build_fp32_model_from_weights(config, fp32_weights_snapshot)
        for i in range(lo, hi):
            qat_q.prepare(surrogate_hybrid.blocks[i])
        surrogate_hybrid.load_state_dict(m_prepared.state_dict())
        surrogate_hybrid = surrogate_hybrid.to(device)
        logger.info("QAT: convert (hybrid, только подготовленные блоки)...")
        convert_hybrid_qat_blocks(m_prepared, qat_q, lo, hi)
        models[labels['hybrid_qat']] = m_prepared
        surrogate_models[labels['hybrid_qat']] = surrogate_hybrid

    # ---------- 7. Сохранение размеров / чекпойнтов (опционально) ----------
    if config['save_models']:
        os.makedirs(config['models_dir'], exist_ok=True)
        for label, m in models.items():
            fname = label.replace(' ', '_').replace('(', '').replace(')', '')
            if label in (labels['full_qat'], labels['hybrid_qat']):
                qpath = os.path.join(config['models_dir'], f"{fname}_converted_state_dict.pth")
                sz_mb = save_state_dict_size_mb(m, qpath)
                logger.info(
                    f"QAT после convert: {label} — сохранён state_dict ({qpath}), размер {sz_mb:.2f} MB"
                )
            else:
                print_model_size(m, fname, config['models_dir'])

    # ---------- 8. Оценка робастности ----------
    results = {}
    for name, model in models.items():
        logger.info(f"\n=== {name} ===")
        model = model.to(device)
        acc_clean = evaluate_clean_accuracy(model, val_loader, device)
        logger.info(f"Clean accuracy: {acc_clean:.4f}")

        # Выбор модели для генерации атаки
        if model_supports_fgsm_input_grad(model, device):
            attack_model = model
            logger.info(f"White-box FGSM доступна для {name} (градиенты есть).")
        else:
            # Для квантованных моделей (после convert) используем подготовленный суррогат
            if name in surrogate_models:
                attack_model = surrogate_models[name]
                logger.info(f"Квантованный суррогат (prepare) используется для генерации атак против {name}.")
            else:
                # Запасной вариант (не должен случаться)
                logger.warning(f"Суррогат для {name} не найден, используется FP32-модель (не рекомендуется).")
                attack_model = model_fp32

        adv_accs = {}
        for eps in config['epsilons']:
            acc_adv = evaluate_adversarial_accuracy(
                target_model=model,
                attack_model=attack_model,
                dataloader=val_loader,
                epsilon=eps,
                mean=mean,
                std=std,
                device=device
            )
            logger.info(f"Epsilon={eps}: {acc_adv:.4f}")
            adv_accs[f"eps={eps}"] = acc_adv

        results[name] = {"clean": acc_clean, **adv_accs}

    # ---------- 9. Сохранение результатов и графиков ----------
    import pandas as pd
    df = pd.DataFrame(results).T
    logger.info("\n=== Результаты (Accuracy) ===")
    print(df.round(4))
    df_sum, df_asr, df_gap = compute_robustness_metric_tables(results, config['epsilons'])
    logger.info("\n=== Сводные метрики (робастность) ===")
    print(df_sum.round(4))
    logger.info("\n=== ASR (attack success rate) по ε ===")
    print(df_asr.round(4))
    logger.info("\n=== Зазор clean − robust по ε ===")
    print(df_gap.round(4))

    save_reproducibility_artifacts(
        config,
        dataset_sizes,
        device,
        results,
        df_results=df,
        df_metrics_summary=df_sum,
        df_asr=df_asr,
        df_gap=df_gap,
    )

    epsilons_plot = [0] + config['epsilons']
    plt.figure(figsize=(10,6))
    for name in results:
        accs = [results[name]["clean"]] + [results[name][f"eps={e}"] for e in config['epsilons']]
        plt.plot(epsilons_plot, accs, marker='o', label=name)
    plt.xlabel("Epsilon (FGSM)")
    plt.ylabel("Accuracy")
    plt.title(
        f"[{config['experiment_variant']}] Робастность ViT (Tiny ImageNet); ветки: {', '.join(branches)}"
    )
    plt.legend()
    plt.grid(True)
    if config['save_models']:
        plt.savefig(os.path.join(config['models_dir'], 'robustness_plot.png'))
    plt.show()
    return config['models_dir'], results

# ============================================
# 10. Функция для общего графика по всем прогонам
# ============================================
def save_combined_robustness_plot(run_summaries, epsilons, save_path, *, xlabel: str, title: str):
    epsilons_plot = [0] + list(epsilons)
    plt.figure(figsize=(11, 6))
    for block in run_summaries:
        tag = block["variant"]
        results = block["results"]
        for name, row in results.items():
            accs = [row["clean"]] + [row[f"eps={e}"] for e in epsilons]
            plt.plot(epsilons_plot, accs, marker="o", linestyle="-", label=f"[{tag}] {name}")
    plt.xlabel(xlabel)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend(fontsize=8, loc="best")
    plt.grid(True)
    plt.tight_layout()
    _d = os.path.dirname(os.path.abspath(save_path))
    if _d:
        os.makedirs(_d, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    logger.info("Сохранён общий график: %s", save_path)

# ============================================
# 11. Основной конвейер (main)
# ============================================
def main():
    """Подряд: FP32 → Full QAT → Hybrid QAT; общие данные, изолированные прогоны.
    Два полных цикла: с adversarial training и бейзлайн без него (оценка влияния квантования).
    """
    cfg_data = dict(CONFIG_SHARED)
    train_loader, val_loader, dataset_sizes, mean, std = download_and_prepare_data(cfg_data)
    dataloaders = {'train': train_loader, 'val': val_loader}

    device_cfg = merge_experiment_config(CONFIG_SHARED, CONFIG_VARIANT_FP32)
    device = prepare_device_for_training(device_cfg)

    def run_three_qat_stages(shared_cfg, series_label, plot_title):
        models_dir_base = shared_cfg['models_dir']
        experiment_run_dirs = []

        cfg = merge_experiment_config(shared_cfg, CONFIG_VARIANT_FP32)
        reset_models_dir_base(cfg, models_dir_base)
        logger.info(f'========== [{series_label}] Прогон 1/3: FP32 ==========')
        rd1, res1 = run_experiment_pipeline(cfg, dataloaders, dataset_sizes, mean, std, device)
        experiment_run_dirs.append(rd1)

        fp32_ckpt = os.path.join(cfg['models_dir'], 'best_model_fp32.pth')
        if not os.path.isfile(fp32_ckpt):
            raise FileNotFoundError(f'После FP32 ожидался чекпойнт: {fp32_ckpt}')

        v_q = dict(CONFIG_VARIANT_FULL_QAT)
        v_q['fp32_checkpoint_path'] = fp32_ckpt
        cfg = merge_experiment_config(shared_cfg, v_q)
        reset_models_dir_base(cfg, models_dir_base)
        logger.info(f'========== [{series_label}] Прогон 2/3: Full QAT ==========')
        rd2, res2 = run_experiment_pipeline(cfg, dataloaders, dataset_sizes, mean, std, device)
        experiment_run_dirs.append(rd2)

        v_hyb = dict(CONFIG_VARIANT_HYBRID_QAT)
        v_hyb['fp32_checkpoint_path'] = fp32_ckpt
        cfg = merge_experiment_config(shared_cfg, v_hyb)
        reset_models_dir_base(cfg, models_dir_base)
        logger.info(f'========== [{series_label}] Прогон 3/3: Hybrid QAT ==========')
        rd3, res3 = run_experiment_pipeline(cfg, dataloaders, dataset_sizes, mean, std, device)
        experiment_run_dirs.append(rd3)

        combined_png = os.path.join(
            os.path.abspath(models_dir_base), f'robustness_all_runs_combined_{series_label}.png'
        )
        save_combined_robustness_plot(
            [
                {"variant": CONFIG_VARIANT_FP32["experiment_variant"], "results": res1},
                {"variant": CONFIG_VARIANT_FULL_QAT["experiment_variant"], "results": res2},
                {"variant": CONFIG_VARIANT_HYBRID_QAT["experiment_variant"], "results": res3},
            ],
            shared_cfg["epsilons"],
            combined_png,
            xlabel="ε (FGSM, L∞ в [0,1])",
            title=plot_title,
        )

        sync_artifacts_to_github(models_dir_base, experiment_run_dirs)

    run_three_qat_stages(
        CONFIG_SHARED,
        'adv',
        'Сводный график: FP32 + Full QAT + Hybrid QAT (Tiny ImageNet), adversarial training',
    )
    if CONFIG_SHARED.get('run_baseline_no_adv', True):
        run_three_qat_stages(
            CONFIG_BASELINE_NO_ADV,
            'no_adv',
            'Сводный график: FP32 + Full QAT + Hybrid QAT (Tiny ImageNet), без adversarial training (бейзлайн)',
        )

# ============================================
# 12. Запуск (одна ячейка ноутбука)
# ============================================
if __name__ == "__main__":
    main()